<div style="background: linear-gradient(90deg, #1a237e 0%, #283593 100%); border-radius: 10px; padding: 18px 28px; margin-bottom: 18px; box-shadow: 0 4px 10px rgba(0,0,0,0.2);">
    <h1 style="margin: 0; color: white; font-size: 26px; font-weight: 700; text-shadow: 1px 1px 2px rgba(0,0,0,0.3);">🔮 PVT (Pyramid Vision Transformer) — RealWaste Classifier</h1>
    <p style="margin: 8px 0 0 0; color: #c5cae9; font-size: 14px;">Advanced PVT backbone · Full evaluation pipeline · Confusion matrix · Per-class diagnostics</p>
</div>

<div style="background: linear-gradient(90deg, #1a237e 0%, #283593 100%); border-radius: 10px; padding: 14px 24px; margin-bottom: 18px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
    <h2 style="margin: 0; color: white; font-size: 22px; font-weight: 700; display: flex; align-items: center; gap: 12px;"><span>📦</span> 1. Install Dependencies</h2>
</div>

In [ ]:
!pip install -q torch torchvision scikit-learn seaborn tqdm
!pip install -q timm==0.9.2  # PVT models are available in timm

<div style="background: linear-gradient(90deg, #1a237e 0%, #283593 100%); border-radius: 10px; padding: 14px 24px; margin-bottom: 18px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
    <h2 style="margin: 0; color: white; font-size: 22px; font-weight: 700; display: flex; align-items: center; gap: 12px;"><span>📚</span> 2. Imports &amp; Seed</h2>
</div>

In [ ]:
import os
import copy
import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('PyTorch:', torch.__version__)
print('timm:', timm.__version__)

<div style="background: linear-gradient(90deg, #1a237e 0%, #283593 100%); border-radius: 10px; padding: 14px 24px; margin-bottom: 18px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
    <h2 style="margin: 0; color: white; font-size: 22px; font-weight: 700; display: flex; align-items: center; gap: 12px;"><span>📂</span> 3. Locate Dataset Automatically</h2>
</div>

In [ ]:
candidate_paths = [
    Path('/kaggle/input/datasets/joebeachcapital/realwaste/realwaste-main/RealWaste'),
    # Add alternate paths here if needed:
    # Path('/kaggle/input/realwaste/RealWaste'),
]

def is_dataset_root(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    class_dirs = [p for p in path.iterdir() if p.is_dir()]
    if len(class_dirs) < 2:
        return False
    has_images = any(
        any(c.glob('*.jpg')) or any(c.glob('*.jpeg')) or any(c.glob('*.png'))
        for c in class_dirs
    )
    return has_images

data_dir = None
for p in candidate_paths:
    if is_dataset_root(p):
        data_dir = p
        break

if data_dir is None:
    raise FileNotFoundError('No valid dataset root found. Update candidate_paths manually.')

print('Using dataset path:', data_dir)

<div style="background: linear-gradient(90deg, #1a237e 0%, #283593 100%); border-radius: 10px; padding: 14px 24px; margin-bottom: 18px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
    <h2 style="margin: 0; color: white; font-size: 22px; font-weight: 700; display: flex; align-items: center; gap: 12px;"><span>📊</span> 4. Build File Index and Inspect Class Balance</h2>
</div>

In [ ]:
valid_ext = {'.jpg', '.jpeg', '.png'}
records = []

class_names = sorted([p.name for p in data_dir.iterdir() if p.is_dir()])
class_to_idx = {c: i for i, c in enumerate(class_names)}

for cls in class_names:
    cls_dir = data_dir / cls
    for img_path in cls_dir.rglob('*'):
        if img_path.suffix.lower() in valid_ext:
            records.append({
                'filepath': str(img_path),
                'label': cls,
                'label_idx': class_to_idx[cls]
            })

df = pd.DataFrame(records)
print('Total images:', len(df))
print('Classes:', class_names)

plt.figure(figsize=(10, 4))
sns.countplot(data=df, x='label', order=class_names)
plt.xticks(rotation=45)
plt.title('Class Distribution (Full Dataset)')
plt.tight_layout()
plt.show()

<div style="background: linear-gradient(90deg, #1a237e 0%, #283593 100%); border-radius: 10px; padding: 14px 24px; margin-bottom: 18px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
    <h2 style="margin: 0; color: white; font-size: 22px; font-weight: 700; display: flex; align-items: center; gap: 12px;"><span>⚙️</span> 5. Stratified Train / Validation / Test Split</h2>
</div>

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df['label_idx'],
    random_state=SEED,
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['label_idx'],
    random_state=SEED,
)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    counts = split['label'].value_counts().sort_index()
    print(f'\n{name} split class counts:')
    print(counts.to_string())

<div style="background: linear-gradient(90deg, #1a237e 0%, #283593 100%); border-radius: 10px; padding: 14px 24px; margin-bottom: 18px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
    <h2 style="margin: 0; color: white; font-size: 22px; font-weight: 700; display: flex; align-items: center; gap: 12px;"><span>🧪</span> 6. Dataset Class and Transforms</h2>
</div>

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_test_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class GarbageDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['filepath']).convert('RGB')
        label = int(row['label_idx'])
        if self.transform:
            image = self.transform(image)
        return image, label


train_ds = GarbageDataset(train_df, transform=train_tfms)
val_ds   = GarbageDataset(val_df,   transform=val_test_tfms)
test_ds  = GarbageDataset(test_df,  transform=val_test_tfms)

print('Datasets ready.')

<div style="background: linear-gradient(90deg, #1a237e 0%, #283593 100%); border-radius: 10px; padding: 14px 24px; margin-bottom: 18px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
    <h2 style="margin: 0; color: white; font-size: 22px; font-weight: 700; display: flex; align-items: center; gap: 12px;"><span>⚖️</span> 7. DataLoaders with Class-Imbalance Handling</h2>
</div>

In [ ]:
train_labels = train_df['label_idx'].values
class_sample_count = np.array([np.sum(train_labels == t) for t in range(len(class_names))])
class_weights = 1.0 / np.maximum(class_sample_count, 1)
sample_weights = class_weights[train_labels]

sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights).double(),
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

print('Loaders ready:', len(train_loader), len(val_loader), len(test_loader))

In [ ]:
def denormalize(img_tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    mean = torch.tensor(mean).view(3, 1, 1)
    std  = torch.tensor(std).view(3, 1, 1)
    return (img_tensor * std + mean).clamp(0, 1)

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flatten()):
    img = denormalize(images[i].cpu())
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(class_names[labels[i].item()])
    ax.axis('off')
plt.suptitle('Sample Training Images')
plt.tight_layout()
plt.show()

<div style="background: linear-gradient(90deg, #1a237e 0%, #283593 100%); border-radius: 10px; padding: 14px 24px; margin-bottom: 18px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
    <h2 style="margin: 0; color: white; font-size: 22px; font-weight: 700; display: flex; align-items: center; gap: 12px;"><span>🔮</span> 8. Build PVT Model</h2>
</div>

Available PVT variants via `timm`:

| Model name (timm) | Params | Notes |
|---|---|---|
| `pvt_v2_b0` | ~3.7 M | Lightest, fastest |
| `pvt_v2_b1` | ~14 M | Good balance |
| `pvt_v2_b2` | ~25 M | Recommended ✅ |
| `pvt_v2_b3` | ~45 M | Heavier, higher accuracy |
| `pvt_v2_b4` | ~63 M | Very heavy |
| `pvt_v2_b5` | ~83 M | Heaviest |

PVTv2 uses overlapping patch embeddings and linear attention — better than plain ViT on dense visual tasks.

In [ ]:
def build_pvt_model(
    backbone: str = 'pvt_v2_b2',
    num_classes: int = 10,
    freeze_features: bool = True,
    pretrained: bool = True,
):
    """
    Build a PVT (Pyramid Vision Transformer) model for image classification.

    Args:
        backbone     : timm model name, e.g. 'pvt_v2_b2'
        num_classes  : number of output classes
        freeze_features : if True, freeze all layers except the classification head
        pretrained   : load ImageNet pre-trained weights

    Returns:
        model (nn.Module)
    """
    model = timm.create_model(
        backbone,
        pretrained=pretrained,
        num_classes=num_classes,
    )

    if freeze_features:
        # Freeze everything, then unfreeze the head
        for p in model.parameters():
            p.requires_grad = False
        # timm PVT models expose the classifier as model.head
        head = getattr(model, 'head', None)
        if head is None:
            head = getattr(model, 'classifier', None)
        if head is not None:
            for p in head.parameters():
                p.requires_grad = True
        else:
            # fallback: unfreeze last Linear layer
            for m in reversed(list(model.modules())):
                if isinstance(m, nn.Linear):
                    for p in m.parameters():
                        p.requires_grad = True
                    break

    return model


# ── Config ──────────────────────────────────────────────────────────────────
BACKBONE = 'pvt_v2_b2'   # Change to pvt_v2_b0 / b1 / b3 etc.

model = build_pvt_model(
    backbone=BACKBONE,
    num_classes=len(class_names),
    freeze_features=True,
    pretrained=True,
).to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params       = sum(p.numel() for p in model.parameters())
print(f'Backbone        : {BACKBONE}')
print(f'Trainable params: {trainable_params:,} / {all_params:,}')
print(f'Frozen params   : {all_params - trainable_params:,}')

<div style="background: linear-gradient(90deg, #1a237e 0%, #283593 100%); border-radius: 10px; padding: 14px 24px; margin-bottom: 18px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
    <h2 style="margin: 0; color: white; font-size: 22px; font-weight: 700; display: flex; align-items: center; gap: 12px;"><span>🛠️</span> 9. Training Utilities</h2>
</div>

In [ ]:
def accuracy_from_logits(logits, targets):
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    losses, accs = [], []
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        logits = model(images)
        # Handle models that return a dict or object
        if hasattr(logits, 'logits'):
            logits = logits.logits
        loss = criterion(logits, targets)
        losses.append(loss.item())
        accs.append(accuracy_from_logits(logits, targets))
    return float(np.mean(losses)), float(np.mean(accs))


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    losses, accs = [], []
    for images, targets in tqdm(loader, leave=False, desc='train'):
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = model(images)
        if hasattr(logits, 'logits'):
            logits = logits.logits
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        accs.append(accuracy_from_logits(logits, targets))
    return float(np.mean(losses)), float(np.mean(accs))


def fit(
    model,
    train_loader,
    val_loader,
    epochs=12,
    lr=1e-3,
    weight_decay=1e-4,
    patience=4,
):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.3, patience=2
    )

    history = []
    best_state    = copy.deepcopy(model.state_dict())
    best_val_loss = float('inf')
    no_improve    = 0

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss,   val_acc   = evaluate(model, val_loader, criterion)
        scheduler.step(val_loss)

        row = {
            'epoch':      epoch,
            'lr':         optimizer.param_groups[0]['lr'],
            'train_loss': train_loss,
            'train_acc':  train_acc,
            'val_loss':   val_loss,
            'val_acc':    val_acc,
        }
        history.append(row)
        print(
            f"Epoch {epoch:02d}/{epochs} | lr {row['lr']:.2e} | "
            f"train_loss {train_loss:.4f} | train_acc {train_acc:.4f} | "
            f"val_loss {val_loss:.4f} | val_acc {val_acc:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = copy.deepcopy(model.state_dict())
            no_improve    = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print('Early stopping triggered.')
                break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

<div style="background: linear-gradient(90deg, #1a237e 0%, #283593 100%); border-radius: 10px; padding: 14px 24px; margin-bottom: 18px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
    <h2 style="margin: 0; color: white; font-size: 22px; font-weight: 700; display: flex; align-items: center; gap: 12px;"><span>🚀</span> 10. Train the Model — Stage 1 (Head Only)</h2>
</div>

Stage 1 trains only the classification head with the PVT backbone frozen. Fast convergence, acts as warm-up.

In [ ]:
EPOCHS_S1 = 10
LR_S1     = 1e-3

model, history_stage1 = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS_S1,
    lr=LR_S1,
    weight_decay=1e-4,
    patience=4,
)

history_stage1.tail()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_stage1['epoch'], history_stage1['train_loss'], label='train_loss')
axes[0].plot(history_stage1['epoch'], history_stage1['val_loss'],   label='val_loss')
axes[0].set_title('Stage 1 Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history_stage1['epoch'], history_stage1['train_acc'], label='train_acc')
axes[1].plot(history_stage1['epoch'], history_stage1['val_acc'],   label='val_acc')
axes[1].set_title('Stage 1 Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

<div style="background: linear-gradient(90deg, #1a237e 0%, #283593 100%); border-radius: 10px; padding: 14px 24px; margin-bottom: 18px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
    <h2 style="margin: 0; color: white; font-size: 22px; font-weight: 700; display: flex; align-items: center; gap: 12px;"><span>🎯</span> 11. Fine-Tuning — Stage 2 (Full Unfreeze)</h2>
</div>

We unfreeze all PVT layers and continue training at a much smaller learning rate for maximum accuracy.

In [ ]:
# Unfreeze the entire model
for p in model.parameters():
    p.requires_grad = True

print('All parameters unfrozen for fine-tuning.')
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {trainable:,}')

model, history_stage2 = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=6,
    lr=1e-4,
    weight_decay=1e-5,
    patience=3,
)

# Merge history
h1 = history_stage1.copy()
h2 = history_stage2.copy()
if len(h2) > 0:
    h2['epoch'] = h2['epoch'] + h1['epoch'].max()
history_all = pd.concat([h1, h2], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_all['epoch'], history_all['train_loss'], label='train_loss')
axes[0].plot(history_all['epoch'], history_all['val_loss'],   label='val_loss')
axes[0].axvline(h1['epoch'].max(), linestyle='--', alpha=0.6, label='stage switch')
axes[0].set_title('Full Training Loss (PVT)')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history_all['epoch'], history_all['train_acc'], label='train_acc')
axes[1].plot(history_all['epoch'], history_all['val_acc'],   label='val_acc')
axes[1].axvline(h1['epoch'].max(), linestyle='--', alpha=0.6, label='stage switch')
axes[1].set_title('Full Training Accuracy (PVT)')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()

<div style="background: linear-gradient(90deg, #1a237e 0%, #283593 100%); border-radius: 10px; padding: 14px 24px; margin-bottom: 18px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
    <h2 style="margin: 0; color: white; font-size: 22px; font-weight: 700; display: flex; align-items: center; gap: 12px;"><span>📈</span> 12. Test Set Evaluation</h2>
</div>

In [ ]:
@torch.no_grad()
def predict_loader(model, loader):
    model.eval()
    all_preds, all_targets = [], []
    for images, targets in tqdm(loader, desc='predicting'):
        images = images.to(device)
        logits = model(images)
        if hasattr(logits, 'logits'):
            logits = logits.logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_targets.extend(targets.numpy().tolist())
    return np.array(all_targets), np.array(all_preds)


y_true, y_pred = predict_loader(model, test_loader)

test_acc = (y_true == y_pred).mean()
print(f'Test Accuracy: {test_acc:.4f}')

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names
)
plt.title(f'Confusion Matrix — {BACKBONE}')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

<div style="background: linear-gradient(90deg, #1a237e 0%, #283593 100%); border-radius: 10px; padding: 14px 24px; margin-bottom: 18px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
    <h2 style="margin: 0; color: white; font-size: 22px; font-weight: 700; display: flex; align-items: center; gap: 12px;"><span>📊</span> 13. Detailed Metrics (Macro, Weighted, Per-Class)</h2>
</div>

In [ ]:
overall_accuracy = accuracy_score(y_true, y_pred)
prec_macro,    rec_macro,    f1_macro,    _ = precision_recall_fscore_support(
    y_true, y_pred, average='macro', zero_division=0
)
prec_weighted, rec_weighted, f1_weighted, _ = precision_recall_fscore_support(
    y_true, y_pred, average='weighted', zero_division=0
)

print('=== Overall Metrics ===')
print(f'Accuracy      : {overall_accuracy:.4f}')
print(f'Precision(Ma) : {prec_macro:.4f}')
print(f'Recall(Ma)    : {rec_macro:.4f}')
print(f'F1-Score(Ma)  : {f1_macro:.4f}')
print(f'Precision(Wt) : {prec_weighted:.4f}')
print(f'Recall(Wt)    : {rec_weighted:.4f}')
print(f'F1-Score(Wt)  : {f1_weighted:.4f}')

p, r, f1, s = precision_recall_fscore_support(
    y_true, y_pred, labels=list(range(len(class_names))), zero_division=0
)

metrics_df = pd.DataFrame({
    'class':     class_names,
    'precision': p,
    'recall':    r,
    'f1_score':  f1,
    'support':   s,
}).sort_values('f1_score', ascending=False)

print('\n=== Per-Class Metrics (sorted by F1) ===')
display(metrics_df.style.format({
    'precision': '{:.4f}',
    'recall':    '{:.4f}',
    'f1_score':  '{:.4f}',
}))

<div style="background: linear-gradient(90deg, #1a237e 0%, #283593 100%); border-radius: 10px; padding: 14px 24px; margin-bottom: 18px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
    <h2 style="margin: 0; color: white; font-size: 22px; font-weight: 700; display: flex; align-items: center; gap: 12px;"><span>🔍</span> 14. Low-F1 Class Diagnostic &amp; Recommendations</h2>
</div>

In [ ]:
threshold_f1  = 0.93
fallback_top_n = 3

low_f1_df  = metrics_df.sort_values('f1_score', ascending=True).copy()
flagged_df = low_f1_df[low_f1_df['f1_score'] < threshold_f1].copy()

if flagged_df.empty:
    flagged_df = low_f1_df.head(fallback_top_n).copy()

suggestion_map = {
    'trash':      'Increase samples + hard augmentations (random crop, blur, brightness/contrast).',
    'plastic':    'Add look-alike negatives (glass/metal) and stronger color/texture augmentation.',
    'metal':      'Add edge/reflective examples; train with higher-resolution images.',
    'paper':      'Add crumpled/wet variants and background diversity.',
    'glass':      'Include transparent/reflective cases with varied lighting.',
    'battery':    'Add close-up and distance variants; balance orientation examples.',
    'cardboard':  'Add folded/torn variants and cluttered backgrounds.',
    'shoes':      'Use occlusion augmentation and mixed-background samples.',
    'biological': 'Include decomposition/shape diversity and lighting changes.',
    'clothes':    'Add texture and color diversity with background clutter.',
}

flagged_df['gap_to_0.95'] = (0.95 - flagged_df['f1_score']).clip(lower=0)
flagged_df['recommended_action'] = flagged_df['class'].map(suggestion_map).fillna(
    'Use class-focused augmentation + collect more hard samples for this class.'
)

print(f'=== Low-F1 Class Diagnostic (threshold: {threshold_f1:.2f}) ===')
print(f'Flagged classes: {len(flagged_df)}')
display(
    flagged_df[['class', 'precision', 'recall', 'f1_score', 'support', 'gap_to_0.95', 'recommended_action']]
    .style
    .format({
        'precision':   '{:.4f}',
        'recall':      '{:.4f}',
        'f1_score':    '{:.4f}',
        'gap_to_0.95': '{:.4f}',
    })
)

print('\nQuick priority order (most urgent first):')
for _, row in flagged_df.sort_values('f1_score').iterrows():
    print(f"  - {row['class']}: F1={row['f1_score']:.4f} | {row['recommended_action']}")

<div style="background: linear-gradient(90deg, #1a237e 0%, #283593 100%); border-radius: 10px; padding: 14px 24px; margin-bottom: 18px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
    <h2 style="margin: 0; color: white; font-size: 22px; font-weight: 700; display: flex; align-items: center; gap: 12px;"><span>🖼️</span> 15. Single-Image Inference Utility</h2>
</div>

In [ ]:
@torch.no_grad()
def predict_image(image_path, model, transform, class_names):
    model.eval()
    img = Image.open(image_path).convert('RGB')
    x = transform(img).unsqueeze(0).to(device)
    logits = model(x)
    if hasattr(logits, 'logits'):
        logits = logits.logits
    probs    = torch.softmax(logits, dim=1).cpu().numpy().squeeze()
    pred_idx = int(np.argmax(probs))
    return class_names[pred_idx], float(probs[pred_idx]), probs


# Example usage (uncomment and edit path):
# sample_path = test_df.iloc[0]['filepath']
# pred_class, confidence, probs = predict_image(sample_path, model, val_test_tfms, class_names)
# print(f'Predicted: {pred_class} | Confidence: {confidence:.4f}')

<div style="background: linear-gradient(90deg, #1a237e 0%, #283593 100%); border-radius: 10px; padding: 14px 24px; margin-bottom: 18px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
    <h2 style="margin: 0; color: white; font-size: 22px; font-weight: 700; display: flex; align-items: center; gap: 12px;"><span>💾</span> 16. Save Artifacts</h2>
</div>

In [ ]:
artifacts_dir = Path('./artifacts')
artifacts_dir.mkdir(parents=True, exist_ok=True)

model_path        = artifacts_dir / f'{BACKBONE}_realwaste_best.pt'
meta_path         = artifacts_dir / 'metadata.json'
class_names_path  = artifacts_dir / 'class_names.json'
frontend_cfg_path = artifacts_dir / 'frontend_config.json'

torch.save({
    'model_state_dict': model.state_dict(),
    'backbone':         BACKBONE,
    'class_names':      class_names,
    'img_size':         IMG_SIZE,
    'mean':             IMAGENET_MEAN,
    'std':              IMAGENET_STD,
}, model_path)

with open(meta_path, 'w') as f:
    json.dump({
        'backbone':      BACKBONE,
        'class_names':   class_names,
        'img_size':      IMG_SIZE,
        'mean':          IMAGENET_MEAN,
        'std':           IMAGENET_STD,
        'test_accuracy': float(test_acc),
        'f1_macro':      float(f1_macro),
    }, f, indent=2)

with open(class_names_path, 'w') as f:
    json.dump(class_names, f, indent=2)

with open(frontend_cfg_path, 'w') as f:
    json.dump({
        'input_type':          'image_file',
        'accepted_extensions': ['jpg', 'jpeg', 'png'],
        'output': {
            'predicted_class': 'string',
            'confidence':      'float',
            'top_k': [{'class_name': 'string', 'probability': 'float'}],
        },
        'normalization': {
            'mean':   IMAGENET_MEAN,
            'std':    IMAGENET_STD,
            'resize': [IMG_SIZE, IMG_SIZE],
        },
    }, f, indent=2)

print('Saved model      :', model_path)
print('Saved metadata   :', meta_path)
print('Saved class names:', class_names_path)
print('Saved frontend   :', frontend_cfg_path)

<div style="background: linear-gradient(90deg, #1a237e 0%, #283593 100%); border-radius: 10px; padding: 14px 24px; margin-bottom: 18px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
    <h2 style="margin: 0; color: white; font-size: 22px; font-weight: 700; display: flex; align-items: center; gap: 12px;"><span>🔬</span> 17. Next Experiments</h2>
</div>

- Try `pvt_v2_b3` or `pvt_v2_b4` for higher accuracy if VRAM permits
- Use cosine annealing scheduler instead of ReduceLROnPlateau
- Add mixup / cutmix augmentation for regularization
- Evaluate test-time augmentation (TTA) — horizontal flip + multi-crop
- Compare PVT vs DeiT side-by-side on this dataset